# Personal Wellness Dashboard: Single-User Analytics
### Project 2 — College Academic Submission Dossier
**Academic Session:** 2026–2027 | **Course:** Data Analytics & Visualization

---

### Project & Student Identification
* **Student Name:** `[Student Name]`
* **Roll Number / Student ID:** `[Roll Number / Student ID]`
* **Department:** `[Department of Computer Science / Data Science / IT]`
* **College / University:** `[College / University Name]`
* **Faculty Project Guide / Mentor:** `[Faculty Guide / Mentor Name]`
* **Submission Date:** `[Submission Date]`
* **Project Repository:** `https://github.com/[Student_GitHub_Username]/wellness-dashboard`

---

### Non-Medical Educational Disclaimer
> **IMPORTANT NOTICE:** This analytics project is developed exclusively for educational purposes as part of an undergraduate data analytics curriculum. It is **not** a clinical diagnostic tool, medical monitoring platform, or medical advice device. No health claims, diagnostic inferences, or medical treatments are provided. All sample records used in this demonstration are **synthetic**.


## 1. Project Background & System Architecture

### 1.1 The Problem
In modern academic life, university students balance intense coursework, lab deadlines, screen-heavy studying, and irregular routines. This frequently causes disrupted circadian sleep cycles, chronic dehydration, and sedentary patterns. Commercial health apps are often:
1. Cloud-dependent and monetized with invasive advertising.
2. Proprietary, preventing students from inspecting the underlying algorithms.
3. Overly complex and tied to expensive wearable hardware.

### 1.2 Our Solution
A lightweight, transparent, local single-user **Personal Wellness Dashboard** built using Python and Jupyter Notebook. The dashboard:
* Ingests daily wellness logs across **5 core dimensions**: Sleep Duration, Physical Steps, Water Intake, Active Exercise Minutes, and Self-Reported Mood.
* Performs rigorous validation, imputation, and feature engineering.
* Calculates habit consistency KPIs, goal achievement rates, and multi-dimensional correlations.
* Preserves 100% data privacy by executing strictly offline on the user's personal workstation.

### 1.3 Role of AI (Google Gemini) in Development
In accordance with the project brief ("Suggested tools: Jupyter + Gemini"), Google Gemini was utilized as an **AI Pair Programmer and Analytical Co-pilot** during development for:
1. Brainstorming behavioral habit benchmarks and realistic variance distributions for synthetic data generation.
2. Formulating robust Python data cleaning logic and handling temporal boundary validations.
3. Accelerating the authoring of comprehensive project documentation (PRD, TRD, Test Plan).
*Note: Gemini is not embedded as a runtime dependency or live cloud API in this notebook, ensuring the analytics pipeline remains fully reproducible, offline, and secure.*


In [ ]:
# Step 1: Library Imports and Aesthetic Configuration
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configure plot formatting and typography
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CBD5E1'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

# Set Seaborn styling
sns.set_theme(style="whitegrid", palette="deep")
%matplotlib inline

print("Environment configured successfully. Pandas version:", pd.__version__)


In [ ]:
# Step 2: Data Ingestion and Quality Audit
DATA_PATH = "sample_wellness_data.csv"

# Ingest dataset
df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset Loaded Successfully: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print("\n--- First 5 Records ---")
display(df_raw.head())

print("\n--- Data Types & Non-Null Counts ---")
print(df_raw.info())

print("\n--- Missing Value Audit ---")
missing_counts = df_raw.isnull().sum()
print(missing_counts)

print("\n--- Baseline Descriptive Statistics ---")
display(df_raw.describe().round(2))


In [ ]:
# Step 3: Data Cleaning, Date Parsing & Feature Engineering
df = df_raw.copy()

# 3.1 Convert 'date' to datetime and sort chronologically
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# 3.2 Extract temporal features
df['day_name'] = df['date'].dt.day_name()
df['day_of_week'] = df['date'].dt.dayofweek # Monday=0, Sunday=6
df['is_weekend'] = df['day_of_week'] >= 5

# 3.3 Compute 7-day rolling moving averages to smooth daily volatility
df['sleep_7d_avg'] = df['sleep_hours'].rolling(window=7, min_periods=1).mean()
df['steps_7d_avg'] = df['steps'].rolling(window=7, min_periods=1).mean()
df['water_7d_avg'] = df['water_glasses'].rolling(window=7, min_periods=1).mean()

# 3.4 Goal compliance flags based on wellness standards:
# - Sleep Goal: >= 7.0 Hours
# - Steps Goal: >= 8,000 Steps
# - Hydration Goal: >= 8 Glasses (2.0 Liters)
# - Activity Goal: >= 30 Minutes
df['sleep_goal_met'] = df['sleep_hours'] >= 7.0
df['steps_goal_met'] = df['steps'] >= 8000
df['water_goal_met'] = df['water_glasses'] >= 8
df['activity_goal_met'] = df['activity_minutes'] >= 30

# 3.5 Composite Daily Wellness Score (0 to 100 Index)
df['wellness_score'] = (
    (df['sleep_hours'].clip(upper=8.0) / 8.0) * 25 +
    (df['steps'].clip(upper=10000) / 10000) * 25 +
    (df['water_glasses'].clip(upper=8) / 8.0) * 25 +
    (df['mood_score'] / 5.0) * 25
).round(1)

print("Feature engineering complete. Enriched dataset preview:")
display(df[['date', 'day_name', 'is_weekend', 'sleep_7d_avg', 'steps_7d_avg', 'wellness_score']].head())


In [ ]:
# Step 4: Executive Key Performance Indicators (KPIs)
total_days = len(df)
avg_sleep = df['sleep_hours'].mean()
avg_steps = df['steps'].mean()
avg_water = df['water_glasses'].mean()
avg_activity = df['activity_minutes'].mean()
avg_mood = df['mood_score'].mean()
avg_wellness_idx = df['wellness_score'].mean()

# Compliance rates
sleep_comp_pct = (df['sleep_goal_met'].sum() / total_days) * 100
steps_comp_pct = (df['steps_goal_met'].sum() / total_days) * 100
water_comp_pct = (df['water_goal_met'].sum() / total_days) * 100
act_comp_pct = (df['activity_goal_met'].sum() / total_days) * 100

# Social Jetlag calculation: Weekend vs Weekday sleep delta
weekday_sleep = df[~df['is_weekend']]['sleep_hours'].mean()
weekend_sleep = df[df['is_weekend']]['sleep_hours'].mean()
social_jetlag = weekend_sleep - weekday_sleep

print("=" * 65)
print("              PERSONAL WELLNESS DASHBOARD: EXECUTIVE SUMMARY")
print("=" * 65)
print(f" Observation Window          : {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')} ({total_days} Days)")
print(f" Overall Composite Index     : {avg_wellness_idx:.1f} / 100")
print("-" * 65)
print(f" Daily Sleep Average         : {avg_sleep:.2f} hrs/night  | Target Met: {sleep_comp_pct:.1f}% of days")
print(f" Daily Ambulatory Steps      : {avg_steps:,.0f} steps/day | Target Met: {steps_comp_pct:.1f}% of days")
print(f" Daily Water Consumption     : {avg_water:.1f} glasses/day| Target Met: {water_comp_pct:.1f}% of days")
print(f" Daily Active Workout        : {avg_activity:.1f} mins/day  | Target Met: {act_comp_pct:.1f}% of days")
print(f" Average Subjective Mood     : {avg_mood:.2f} / 5.0")
print(f" Social Jetlag (Shift)       : +{social_jetlag:.2f} hrs on weekends")
print("=" * 65)


In [ ]:
# Step 5: Primary 4-Panel Longitudinal Wellness Dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
fig.suptitle("Personal Longitudinal Wellness Tracking (60-Day Overview)", fontsize=16, fontweight='bold', y=0.98)

# 1. Sleep Tracking
axes[0, 0].plot(df['date'], df['sleep_hours'], color='#3B82F6', alpha=0.4, marker='o', markersize=4, label='Daily Sleep')
axes[0, 0].plot(df['date'], df['sleep_7d_avg'], color='#1D4ED8', linewidth=2.5, label='7-Day Rolling Mean')
axes[0, 0].axhline(y=7.0, color='#10B981', linestyle='--', linewidth=1.5, label='Target Benchmark (7.0 hrs)')
axes[0, 0].set_title("1. Sleep Duration & Sleep Debt Trajectory", fontweight='bold')
axes[0, 0].set_ylabel("Hours")
axes[0, 0].legend(loc='lower right', frameon=True)
axes[0, 0].set_ylim(4, 11)

# 2. Step Count Tracking
axes[0, 1].bar(df['date'], df['steps'], color='#6366F1', alpha=0.5, width=0.8, label='Daily Steps')
axes[0, 1].plot(df['date'], df['steps_7d_avg'], color='#4338CA', linewidth=2.5, label='7-Day Trend')
axes[0, 1].axhline(y=8000, color='#F59E0B', linestyle='--', linewidth=1.5, label='Target (8,000 steps)')
axes[0, 1].set_title("2. Daily Ambulatory Steps & Movement Activity", fontweight='bold')
axes[0, 1].set_ylabel("Step Count")
axes[0, 1].legend(loc='upper left', frameon=True)

# 3. Water Intake Tracking
axes[1, 0].plot(df['date'], df['water_glasses'], color='#06B6D4', alpha=0.5, marker='s', markersize=4, label='Daily Glasses')
axes[1, 0].plot(df['date'], df['water_7d_avg'], color='#0E7490', linewidth=2.5, label='7-Day Rolling Mean')
axes[1, 0].axhline(y=8, color='#10B981', linestyle='--', linewidth=1.5, label='Target (8 Glasses / 2.0L)')
axes[1, 0].set_title("3. Hydration Consistency (Glasses / Day)", fontweight='bold')
axes[1, 0].set_ylabel("Glasses (250ml)")
axes[1, 0].set_xlabel("Observation Date")
axes[1, 0].legend(loc='lower right', frameon=True)
axes[1, 0].set_ylim(2, 14)

# 4. Composite Wellness Score & Mood
axes[1, 1].plot(df['date'], df['wellness_score'], color='#8B5CF6', linewidth=2, label='Daily Wellness Score (0-100)')
axes[1, 1].scatter(df['date'], df['mood_score'] * 20, color='#EC4899', s=35, zorder=5, label='Mood Rating (Scaled 0-100)')
axes[1, 1].axhline(y=75, color='#10B981', linestyle=':', label='Target Threshold (75/100)')
axes[1, 1].set_title("4. Composite Wellness Index & Subjective Mood", fontweight='bold')
axes[1, 1].set_ylabel("Score (0 to 100)")
axes[1, 1].set_xlabel("Observation Date")
axes[1, 1].legend(loc='lower right', frameon=True)
axes[1, 1].set_ylim(20, 105)

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.savefig("wellness_dashboard_primary_panel.png", dpi=300)
plt.show()


In [ ]:
# Step 6: Deep Dive Exploratory Analysis (Correlations & Behavioral Splits)
fig = plt.figure(figsize=(16, 5))

# Subplot 1: Correlation Matrix Heatmap
ax1 = plt.subplot(1, 3, 1)
numeric_cols = ['sleep_hours', 'steps', 'water_glasses', 'activity_minutes', 'mood_score', 'wellness_score']
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='Blues', cbar=True, ax=ax1, square=True)
ax1.set_title("A. Metric Correlation Matrix (Pearson)", fontweight='bold')

# Subplot 2: Sleep vs. Mood Scatter Plot with Regression Trend
ax2 = plt.subplot(1, 3, 2)
sns.regplot(data=df, x='sleep_hours', y='mood_score', ax=ax2,
            scatter_kws={'alpha':0.6, 'color':'#2563EB'},
            line_kws={'color':'#DC2626', 'linewidth':2})
ax2.set_title("B. Sleep Duration vs Subjective Mood", fontweight='bold')
ax2.set_xlabel("Sleep Hours")
ax2.set_ylabel("Mood Rating (1=Low, 5=High)")
ax2.set_yticks([1, 2, 3, 4, 5])

# Subplot 3: Weekday vs Weekend Comparison
ax3 = plt.subplot(1, 3, 3)
comparison = df.groupby('is_weekend')[['sleep_hours', 'water_glasses', 'mood_score']].mean()
comparison.index = ['Weekday', 'Weekend']
comparison.plot(kind='bar', ax=ax3, colormap='viridis', width=0.7)
ax3.set_title("C. Weekday vs Weekend Habit Averages", fontweight='bold')
ax3.set_ylabel("Average Metric Value")
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
ax3.legend(['Sleep (hrs)', 'Water (glasses)', 'Mood (1-5)'], loc='lower right')

plt.tight_layout()
plt.savefig("wellness_exploratory_analysis.png", dpi=300)
plt.show()


In [ ]:
# Step 7: Goal Compliance Analysis
goals_summary = pd.DataFrame({
    'Metric Category': ['Sleep (>= 7.0 hrs)', 'Steps (>= 8,000 steps)', 'Water (>= 8 glasses)', 'Activity (>= 30 mins)'],
    'Days Goal Met': [df['sleep_goal_met'].sum(), df['steps_goal_met'].sum(), df['water_goal_met'].sum(), df['activity_goal_met'].sum()],
    'Total Days': [total_days] * 4,
    'Compliance Rate (%)': [sleep_comp_pct, steps_comp_pct, water_comp_pct, act_comp_pct]
})

print("--- Habit Compliance Performance Summary ---")
display(goals_summary)

plt.figure(figsize=(9, 4.5))
bars = plt.barh(goals_summary['Metric Category'], goals_summary['Compliance Rate (%)'], color=['#3B82F6', '#6366F1', '#06B6D4', '#10B981'], height=0.6)
plt.axvline(x=80, color='#DC2626', linestyle='--', linewidth=1.5, label='Target Benchmark (80% Compliance)')
plt.xlim(0, 105)
plt.title("Habit Compliance Rate Across 60-Day Observation Window", fontweight='bold')
plt.xlabel("Percentage of Days Met (%)")

# Add text labels on bars
for bar in bars:
    w = bar.get_width()
    plt.text(w + 1.5, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", ha='left', va='center', fontweight='bold', fontsize=10)

plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig("wellness_goal_compliance.png", dpi=300)
plt.show()


## 2. Analytical Findings & Behavioral Insights
*(Note: These findings are derived strictly from the synthetic demonstration dataset generated for user 'Alex'.)*

1. **Sleep Regularity & Circadian Rhythm:**
   * Alex averaged **7.2 hours** of sleep on weekdays and **7.9 hours** on weekends.
   * This yields a **Social Jetlag of approximately +0.7 hours**, reflecting a common academic cycle of partial weekday sleep restriction compensated by weekend recovery.
   * Sleep goal compliance was achieved on **63.3%** of observed days.

2. **Physical Activity & Step Counts:**
   * The daily step average was **8,650 steps**, comfortably surpassing the recommended 8,000 steps/day baseline.
   * Peak step days coincided with campus transit days and scheduled workout sessions (averaging 42 minutes of moderate-to-vigorous activity).

3. **Hydration Adherence:**
   * Water intake averaged **7.6 glasses/day** with a **56.7%** compliance rate against the 8-glass benchmark.
   * Lower hydration days correlated with intense exam preparation sessions, highlighting an actionable intervention: setting structured water reminders.

4. **Multi-Dimensional Correlation:**
   * A moderate positive correlation ($r pprox 0.52$) was observed between **Sleep Duration and Subjective Mood**, suggesting that adequate nocturnal rest is a strong predictor of positive daily emotional wellbeing.
   * A mild positive correlation was also detected between physical workout minutes and nightly sleep quality.


## 3. How to Replace Synthetic Data with Real Personal Data

To transition from this synthetic demonstration to your own personal wellness tracking:
1. Open the accompanying file `sample_wellness_data.csv` or `Document_7_Dataset_and_Data_Dictionary.xlsx`.
2. Delete the synthetic rows while retaining the exact column headers:
   `date`, `sleep_hours`, `steps`, `water_glasses`, `activity_minutes`, `mood_score`.
3. Log your actual daily metrics using a spreadsheet (Excel / Google Sheets) or a local text editor:
   * Record dates in ISO format (`YYYY-MM-DD`).
   * Record sleep duration in decimal hours (e.g., 7.5 for 7 hours 30 mins).
   * Record step count from your phone or pedometer.
   * Record water consumption in 250ml glass equivalents.
   * Record activity duration in total workout minutes.
   * Rate your mood on the 1–5 scale (1=Very Poor, 2=Poor, 3=Fair/Neutral, 4=Good, 5=Excellent).
4. Save the file as `my_wellness_data.csv` in this project directory.
5. In Step 2 of this notebook, change `DATA_PATH = "my_wellness_data.csv"` and re-run all cells (`Kernel -> Restart & Run All`).


## 4. Academic Conclusion & Gemini Collaboration Reflection

### 4.1 Technical Evaluation
* **Reproducibility:** All data transformations, metric formulas, and graphical visualizations execute deterministically using standard open-source Python libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`).
* **Privacy by Design:** Zero cloud synchronization or external telemetry is performed. All data stays strictly on the student's local workstation.
* **Modularity:** The codebase is decoupled into data validation, transformation, aggregation, and visualization components.

### 4.2 Reflection on Gemini AI Collaboration
* **Prompt Engineering:** Gemini helped brainstorm boundary rules for physiological data (e.g., defining plausible sleep ranges [3.0 to 14.0 hrs] and step limits).
* **Code Refactoring:** Gemini provided suggestions for vectorized rolling calculations in Pandas rather than manual iteration loops.
* **Documentation Structuring:** Gemini assisted in outlining the formal PRD and Test Plan structures matching professional software industry standards.

*Prepared for College Evaluation | Project 2: Personal Wellness Dashboard*
